# Lección 5.2 — Regresión logística: de la recta a la probabilidad

> **Módulo 5 · Modelos clásicos supervisados** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 110–130 min (lección con ejercicios intercalados ≈ 85 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 5.1 (regresión lineal), Lección 3.2 (entropía cruzada), Lección 4.2 (precisión y *recall*), Lección 4.3 (umbrales)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar cómo una regresión logística convierte una suma ponderada en una **probabilidad**.
- Describir **cómo aprende**: castigando las predicciones seguras pero equivocadas.
- Usar `predict_proba` y **mover el umbral** de decisión según el costo de cada error.
- Predecir el efecto de sus hiperparámetros (`C`, `class_weight`, `max_iter`).
- Reconocer cuándo es una buena elección y cuándo no.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.datasets import make_moons, make_circles, load_breast_cancer

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el pronóstico del clima

La app del clima no dice *"mañana llueve"*. Dice **"70 % de probabilidad de lluvia"**. Y tú decides:
- Si solo vas a tender la ropa, con 70 % quizá no te arriesgas.
- Si vas a una boda al aire libre, ¡con 30 % ya rentas una carpa!

**La misma probabilidad lleva a decisiones distintas según lo que cueste equivocarse.**

Una regresión logística hace eso: en lugar de responder "sí" o "no", da una **probabilidad**. La decisión final (el **umbral**) la pones tú.

> 📝 A pesar de su nombre, la regresión logística es un modelo de **clasificación**. Se llama "regresión" porque por dentro calcula un número (como la regresión lineal de la Lección 5.1) y luego lo convierte en probabilidad.

### ✋ Ejercicio 1 — ¿Por qué no basta una recta?
Queremos predecir si un estudiante **aprueba** (1) o **reprueba** (0) según sus horas de estudio. Alguien propone usar la regresión lineal de la Lección 5.1: `predicción = −0.1 + 0.15 × horas`.

1. ¿Qué "probabilidad" da para **0 horas**? ¿Y para **12 horas**?
2. ¿Qué problema ves?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. 0 horas → **−0.10**; 12 horas → **1.70**.
2. ¡Una probabilidad no puede ser negativa ni mayor que 1 (100 %)! Una recta crece sin límite hacia los dos lados. Necesitamos algo que **aplaste** la recta para que siempre quede entre 0 y 1.
</details>

---
## 2. 📖 El concepto: una recta "aplastada" por la sigmoide

La regresión logística trabaja en dos pasos:

```
 1) Puntaje = b + w₁·x₁ + w₂·x₂ + …        (igual que la regresión lineal: va de −∞ a +∞)
 2) Probabilidad = sigmoide(puntaje)       (lo aplasta entre 0 y 1)
```

La **sigmoide** es una curva en forma de **S**:
- puntajes muy negativos → probabilidad cercana a **0**;
- puntajes muy positivos → probabilidad cercana a **1**;
- y en medio, una transición suave.

In [ ]:
def sigmoide(puntaje):
    return 1 / (1 + np.exp(-puntaje))

for p in [-6, -2, -1, 0, 1, 2, 6]:
    print(f"puntaje {p:>3}  →  probabilidad {sigmoide(p):.3f}")

xs = np.linspace(-8, 8, 200)
plt.plot(xs, sigmoide(xs)); plt.axhline(0.5, ls="--", c="gray")
plt.xlabel("Puntaje (b + w·x)"); plt.ylabel("Probabilidad"); plt.title("La sigmoide"); plt.show()

### 🔮 Ejercicio 2 — El punto de equilibrio
Mirando la tabla de arriba:
1. ¿Qué **puntaje** da exactamente 50 % de probabilidad?
2. Si el puntaje sube de 6 a 10, ¿la probabilidad cambia mucho o poco? ¿Por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Un puntaje de **0** → 50 %. Puntajes positivos → más de 50 %; negativos → menos de 50 %.
2. **Muy poco** (de 0.998 a 0.99995). En los extremos la S está casi plana: el modelo ya está casi seguro y más evidencia apenas cambia la probabilidad.
</details>

### 📖 Con datos reales de estudiantes
Ahora tenemos 200 estudiantes con sus horas de estudio y si aprobaron. Entrenamos una `LogisticRegression` y le pedimos **probabilidades** con `predict_proba`.

> 📝 `predict_proba` devuelve dos columnas: la probabilidad de la clase 0 y la de la clase 1 (suman 1). Casi siempre nos interesa la segunda: `predict_proba(X)[:, 1]`.

In [ ]:
rng = np.random.default_rng(0)
horas = rng.uniform(0, 10, 200).round(1)
aprobo = (rng.random(200) < sigmoide(-4 + 0.9 * horas)).astype(int)    # regla oculta
estudiantes = pd.DataFrame({"horas": horas, "aprobo": aprobo})

logit = LogisticRegression().fit(estudiantes[["horas"]], estudiantes.aprobo)
print(f"Intercepto b = {logit.intercept_[0]:.2f}   ·   Coeficiente w = {logit.coef_[0, 0]:.2f}")

nuevos = pd.DataFrame({"horas": [1, 3, 5, 7, 9]})
nuevos["prob_aprobar"] = logit.predict_proba(nuevos)[:, 1].round(2)
nuevos["predict"] = logit.predict(nuevos[["horas"]])
nuevos

In [ ]:
xs_h = pd.DataFrame({"horas": np.linspace(0, 10, 200)})
plt.scatter(estudiantes.horas, estudiantes.aprobo + rng.normal(0, 0.02, 200), s=10, alpha=.5, label="Estudiantes (0 = reprobó, 1 = aprobó)")
plt.plot(xs_h.horas, logit.predict_proba(xs_h)[:, 1], "r-", lw=2, label="Probabilidad de aprobar")
plt.axhline(0.5, ls="--", c="gray"); plt.xlabel("Horas de estudio"); plt.legend(); plt.show()

### ✋ Ejercicio 3 — Lee el modelo
1. El coeficiente $w$ es positivo. ¿Qué significa en palabras?
2. Mira la tabla: ¿qué relación hay entre la columna `prob_aprobar` y la columna `predict`?
3. 🔮 ¿A partir de **cuántas horas** el modelo predice "aprueba"? Pista: es donde el puntaje vale 0, es decir, $b + w \cdot \text{horas} = 0$. Calcula $-b / w$ y compruébalo con la celda de abajo.

✍️ *Mis respuestas:*

In [ ]:
print(f"Frontera: {-logit.intercept_[0] / logit.coef_[0, 0]:.2f} horas")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Más horas → mayor probabilidad de aprobar.** (Un coeficiente negativo bajaría la probabilidad.) Ojo: a diferencia de la regresión lineal, el coeficiente **no** es "cuántos puntos de probabilidad suma cada hora", porque la S no es una recta: una hora extra cambia mucho la probabilidad cerca de la frontera y casi nada en los extremos.
2. `predict` es simplemente **`prob_aprobar ≥ 0.5`**. Por defecto, scikit-learn usa un **umbral de 50 %**.
3. **5 horas** ($-b/w = 4.80 / 0.96$): justo donde la tabla da `prob_aprobar` = 0.50. Con menos, la probabilidad es menor que 50 % y predice "reprueba"; con más, "aprueba".
</details>

---
## 3. 📖 Cómo aprende: castigar la seguridad equivocada

Para aprender, el modelo necesita medir qué tan mal lo hace. Usa la **entropía cruzada** o **pérdida logarítmica** (*log loss*, Lección 3.2). La idea:

| El modelo le dio a la clase **correcta** una probabilidad de… | Castigo |
|---|---|
| 0.99 (seguro y acertó) | casi 0 |
| 0.60 (dudoso y acertó) | pequeño |
| 0.10 (bastante seguro… y falló) | grande |
| 0.01 (segurísimo… y falló) | **enorme** |

Equivocarse con dudas se perdona; **equivocarse con mucha seguridad se castiga muchísimo**.

Luego, igual que el excursionista de la Lección 3.3, el modelo ajusta $b$ y $w$ poco a poco (con un algoritmo de optimización, el `solver`) hasta que la pérdida total es mínima. A diferencia de la regresión lineal, **no hay fórmula directa**: necesita iterar (de ahí el hiperparámetro `max_iter`).

### 🔮 Ejercicio 4 — ¿Cuánto castigo?
Un estudiante **sí aprobó**. Tres modelos le dieron estas probabilidades de aprobar: **A = 0.9**, **B = 0.5**, **C = 0.05**.
1. Ordénalos del menor al mayor castigo.
2. ¿El castigo de C será unas 2 veces el de B, o mucho más?

✍️ *Mi predicción:*

In [ ]:
for nombre, p in {"A": 0.9, "B": 0.5, "C": 0.05}.items():     # ✏️ prueba con 0.001
    print(f"Modelo {nombre}: probabilidad a la clase correcta = {p:<5} → castigo = {-np.log(p):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **A (0.11) < B (0.69) < C (3.00).**
2. Mucho más: C recibe **más de 4 veces** el castigo de B. Y si pruebas con 0.001, el castigo sube a 6.9. La pérdida logarítmica crece sin límite cuando el modelo está seguro de algo falso. Por eso la regresión logística tiende a dar probabilidades **bien calibradas**: exagerar la seguridad sale caro.
</details>

---
## 4. 📖 ¿Qué pasa si las clases no se separan con una recta?

Con dos características, la frontera de decisión (donde la probabilidad es 50 %) es el lugar donde el puntaje vale 0: $b + w_1 x_1 + w_2 x_2 = 0$. Eso es **una línea recta**. La regresión logística es un **clasificador lineal**.

¿Qué pasa con datos que no se separan con una recta?

In [ ]:
def dibujar_frontera(modelo, X, y, ax, titulo):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 300),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 300))
    zz = modelo.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=20, alpha=.4, cmap="coolwarm")
    ax.contour(xx, yy, zz, levels=[0.5], colors="k")
    ax.scatter(*X.T, c=y, cmap="coolwarm", edgecolor="k", s=15)
    ax.set_title(titulo, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

X_lunas, y_lunas = make_moons(n_samples=300, noise=0.2, random_state=0)
X_circ, y_circ = make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=0)

### 🔮 Ejercicio 5 — Lunas y círculos
Vamos a entrenar una regresión logística en dos conjuntos: **dos lunas** y **dos círculos concéntricos** (uno dentro del otro).
¿Qué exactitud (validación cruzada) esperas en cada uno: alta, media o como lanzar una moneda?

✍️ *Mis predicciones:*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (X_, y_, nombre) in zip(axes, [(X_lunas, y_lunas, "Lunas"), (X_circ, y_circ, "Círculos")]):
    m = LogisticRegression().fit(X_, y_)
    cv = cross_val_score(LogisticRegression(), X_, y_, cv=5).mean()
    dibujar_frontera(m, X_, y_, ax, f"{nombre}: exactitud CV = {cv:.0%}")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Lunas:** ≈84 %. Una recta separa "bastante bien" las dos lunas, aunque se equivoca en las puntas.
- **Círculos:** ≈46 %, ¡como lanzar una moneda! No existe ninguna recta que separe un círculo de otro que lo rodea.

Fíjate en el **fondo degradado**: no es solo rojo o azul, sino la **probabilidad**. Cerca de la frontera el modelo duda (colores pálidos); lejos, está seguro.
</details>

### 📖 El mismo truco de la Lección 5.1
Igual que con la curva de consumo, podemos **inventar características**. Con $x_1^2$ y $x_2^2$ el modelo puede dibujar una frontera circular, aunque por dentro siga siendo "lineal en sus coeficientes".

In [ ]:
GRADO = 2      # ✏️ prueba 1 y 3
circ_poli = make_pipeline(PolynomialFeatures(GRADO), StandardScaler(), LogisticRegression())
cv = cross_val_score(circ_poli, X_circ, y_circ, cv=5).mean()

fig, ax = plt.subplots(figsize=(4.5, 4))
dibujar_frontera(circ_poli.fit(X_circ, y_circ), X_circ, y_circ, ax, f"Círculos con grado {GRADO}: CV = {cv:.0%}")
plt.show()

Con `GRADO = 2` la exactitud salta a ≈100 %. Pero inventar características a mano solo funciona si **adivinas** la forma correcta. En las próximas lecciones verás modelos que encuentran fronteras curvas por sí solos: KNN (5.3), árboles (5.4) y SVM con *kernel* (5.5).

---
## 5. 📖 Probabilidades y umbral: la decisión es tuya

Esta es **la** gran ventaja práctica de la regresión logística: te da una probabilidad, y tú decides a partir de qué probabilidad actuar.

```
  predict_proba  →  0.03   0.18   0.42   0.61   0.95
  umbral 0.5     →   no     no     no     SÍ     SÍ      (lo que hace predict)
  umbral 0.2     →   no     no     SÍ     SÍ     SÍ      (más alertas)
```

- **Bajar el umbral** → más casos marcados como positivos → **sube el *recall*** (atrapas más), **baja la precisión** (más falsas alarmas).
- **Subir el umbral** → al revés.

Es el compromiso que viste en la Lección 4.3. Vamos a verlo con datos médicos reales: tumores de mama, donde **1 = maligno**.

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
yc = 1 - yc        # en el original 0 = maligno; lo invertimos para que 1 = maligno (lo que queremos detectar)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=0)

modelo_med = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xc_tr, yc_tr)
prob_maligno = modelo_med.predict_proba(Xc_te)[:, 1]

print(f"Pacientes de prueba: {len(yc_te)}  ·  malignos: {yc_te.sum()}")
print("Probabilidad de 'maligno' de los primeros 8 pacientes:", prob_maligno[:8].round(3))

### 🔮 Ejercicio 6 — Mueve el umbral
La celda de abajo aplica tres umbrales: 0.5, 0.3 y 0.1. Antes de ejecutar:
1. ¿Con cuál detectarás **más** tumores malignos (*recall*)?
2. ¿Con cuál tendrás **más falsas alarmas** (sanos marcados como malignos)?

✍️ *Mis predicciones:*

In [ ]:
for umbral in [0.5, 0.3, 0.1]:          # ✏️ prueba otros umbrales
    pred = (prob_maligno >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(yc_te, pred).ravel()
    print(f"umbral {umbral:<4} → malignos detectados {tp}/{tp + fn} (recall {recall_score(yc_te, pred):.0%})  "
          f"· falsas alarmas {fp:>2}  · precisión {precision_score(yc_te, pred):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. El umbral **0.1** detecta más: los **53 de 53** malignos (con 0.5 y 0.3 se escapan 2).
2. También es el que da más falsas alarmas: de **0** con umbral 0.5 a **11** con 0.1 (la precisión baja de 100 % a 83 %).

En medicina, dejar pasar un cáncer (falso negativo) es muchísimo peor que pedir una biopsia de más (falso positivo). Por eso aquí **tiene sentido bajar el umbral**, aunque baje la precisión. Con `predict` a secas (umbral 0.5) nunca habrías tenido esa opción.
</details>

### 📖 ¿Qué umbral elegir? Deja que lo decidan los costos
Supón que un falso negativo cuesta $C_{FN}$ y un falso positivo $C_{FP}$. Para un caso con probabilidad $p$ de ser positivo:
- Si lo marcas **negativo**, te equivocas con probabilidad $p$ → costo esperado $C_{FN} \cdot p$.
- Si lo marcas **positivo**, te equivocas con probabilidad $1-p$ → costo esperado $C_{FP} \cdot (1-p)$.

Conviene marcarlo positivo cuando $C_{FN} \cdot p > C_{FP} \cdot (1-p)$. Despejando:

$$\text{umbral} = \frac{C_{FP}}{C_{FP} + C_{FN}}$$

Ejemplo: si un falso negativo cuesta **4 veces** más que un falso positivo ($C_{FN}=4$, $C_{FP}=1$), el umbral es $1/(1+4) = 0.2$.

> ⚠️ Esto solo funciona si las probabilidades están **bien calibradas** (si "30 %" de verdad significa que 3 de cada 10 casos así son positivos). La regresión logística suele estarlo; otros modelos, como Naive Bayes (Lección 5.6), no tanto.

### ✋ Ejercicio 7 — Calcula el umbral
1. Una tienda en línea revisa a mano los pedidos sospechosos de fraude. Revisar un pedido legítimo cuesta **$1**; dejar pasar un fraude cuesta **$9**. ¿Qué umbral usarías?
2. Un banco envía un SMS de "¿reconoces esta compra?". Molestar a un cliente honesto cuesta **2**; no avisar de un fraude cuesta **2**. ¿Umbral?
3. ¿Qué pasa con el umbral cuando el falso negativo se vuelve cada vez más caro?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. $1 / (1 + 9) =$ **0.1**: revisa todo pedido con más de 10 % de probabilidad de fraude.
2. $2 / (2 + 2) =$ **0.5**: cuando los dos errores cuestan lo mismo, el umbral por defecto es el correcto. ¡Ese es el supuesto oculto de `predict`!
3. **Baja** hacia 0: cuanto más caro es dejar pasar un positivo, menos evidencia necesitas para actuar.
</details>

### 📖 Elegir el umbral sin hacer trampa: `cross_val_predict`
El umbral teórico es un gran punto de partida, pero conviene comprobarlo con datos. **Nunca** con el conjunto de prueba (Lección 4.6): usamos **validación cruzada** dentro del entrenamiento.

`cross_val_predict(modelo, X, y, cv=5, method="predict_proba")` divide el entrenamiento en 5 pliegues y, para cada ejemplo, devuelve la probabilidad que le dio un modelo **que no lo vio** al entrenar. Con esas probabilidades "honestas" podemos probar todos los umbrales que queramos.

Lo probamos con un caso de negocio: una tienda en línea con 2,000 transacciones, de las cuales solo unas pocas son **fraude** (1). Dejar pasar un fraude cuesta **10**; revisar a mano una compra legítima cuesta **1**.

In [ ]:
rng_f = np.random.default_rng(5)
n_f = 2_000
trans = pd.DataFrame({
    "monto":         rng_f.exponential(1_000, n_f).round(),
    "hora":          rng_f.integers(0, 24, n_f),
    "pais_distinto": rng_f.integers(0, 2, n_f),
})
riesgo = -5.2 + 0.0008 * trans.monto + 1.2 * trans.pais_distinto + 0.8 * (trans.hora < 6)
trans["fraude"] = (rng_f.random(n_f) < sigmoide(riesgo)).astype(int)
Xf, yf = trans.drop(columns="fraude"), trans.fraude
print(f"Fraudes: {yf.mean():.1%}")

COSTO_FN, COSTO_FP = 10, 1        # ✏️ cambia los costos

def costo_promedio(y_real, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return (COSTO_FN * fn + COSTO_FP * fp) / len(y_real)

# En un caso real, aquí usarías solo tu conjunto de entrenamiento
prob_cv = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()),
                            Xf, yf, cv=5, method="predict_proba")[:, 1]

umbrales = np.arange(0.02, 0.91, 0.02)
costos = [costo_promedio(yf, (prob_cv >= u).astype(int)) for u in umbrales]
mejor = umbrales[np.argmin(costos)]
teorico = COSTO_FP / (COSTO_FP + COSTO_FN)

plt.plot(umbrales, costos, "o-", ms=3)
plt.axvline(teorico, ls="--", c="g", label=f"Umbral teórico = {teorico:.2f}")
plt.axvline(0.5, ls=":", c="gray", label="Umbral por defecto = 0.5")
plt.xlabel("Umbral"); plt.ylabel("Costo promedio por transacción"); plt.legend(); plt.show()
print(f"Costo con umbral 0.5      : {costo_promedio(yf, (prob_cv >= 0.5).astype(int)):.3f}")
print(f"Costo con umbral teórico  : {costo_promedio(yf, (prob_cv >= teorico).astype(int)):.3f}")
print(f"Mejor umbral en la gráfica: {mejor:.2f} → costo {min(costos):.3f}")

### ✋ Ejercicio 8 — Lee la curva
1. ¿Cuánto se reduce el costo al pasar del umbral 0.5 al teórico?
2. El "mejor umbral de la gráfica" no coincide exactamente con el teórico. ¿Cuál usarías y por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. De ≈0.38 a ≈0.33 por transacción (umbral teórico $1/11 ≈ 0.09$): **≈13 % menos** de costo, sin cambiar de modelo. Solo moviendo el umbral.
2. El mínimo de la gráfica (≈0.14, costo ≈0.31) está cerca y cuesta casi lo mismo: la curva es bastante plana en esa zona, y con solo ~80 fraudes en los datos unos pocos casos mueven el mínimo exacto, que puede ser **ruido**. El teórico tiene una ventaja enorme: **se deriva de la regla de negocio** y es fácil de defender ("revisamos cuando el riesgo supera 1 de cada 11"). Un umbral cercano al teórico y comprobado con validación cruzada es una gran elección.
</details>

---
## 6. 📖 Hiperparámetros

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `C` | **Inverso** de la regularización (Lección 7.1): cuánta libertad tienen los coeficientes | Menos castigo → coeficientes grandes, probabilidades extremas, riesgo de **sobreajuste** | Más castigo → coeficientes pequeños, probabilidades cercanas a 0.5, riesgo de **subajuste** | 1.0 (probar 0.01–100 con validación cruzada) |
| `class_weight` | Cuánto pesa cada clase en la pérdida | — | — | `None`; `"balanced"` si la clase que te importa es minoritaria |
| `max_iter` | Pasos máximos del optimizador | Más tiempo para converger | Puede detenerse antes del fondo (`ConvergenceWarning`) | 100; súbelo (1000) si aparece el aviso |
| `l1_ratio` | Tipo de castigo: 0 = L2 (*Ridge*), 1 = L1 (*Lasso*, pone coeficientes en 0) | Más coeficientes exactamente en 0 | Castigo suave a todos | 0 (L2) |
| `solver` | Algoritmo de optimización | — | — | `lbfgs` (por defecto) |

> ⚠️ **Ojo con `C`:** funciona **al revés** que el `alpha` de `Ridge` en la Lección 5.1. `C` grande = **poca** regularización.

### 🔮 Ejercicio 9 — La perilla `C`
Entrenamos el modelo de los estudiantes con `C` = 0.001, 1 y 1000. ¿Qué crees que pasa con la probabilidad que le da a alguien que estudió **9 horas**? ¿Y con la forma de la S?

✍️ *Mi predicción:*

In [ ]:
for c in [0.001, 1, 1000]:          # ✏️ prueba otros valores
    m = LogisticRegression(C=c).fit(estudiantes[["horas"]], estudiantes.aprobo)
    p9 = m.predict_proba(pd.DataFrame({"horas": [9]}))[0, 1]
    plt.plot(xs_h.horas, m.predict_proba(xs_h)[:, 1], lw=2, label=f"C={c}  (w={m.coef_[0, 0]:.2f}, P(9 h)={p9:.2f})")
plt.axhline(0.5, ls="--", c="gray"); plt.xlabel("Horas de estudio"); plt.ylabel("Probabilidad de aprobar")
plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- `C=0.001` → castigo fortísimo: el coeficiente se encoge (w ≈ 0.16 en lugar de ≈ 0.96) y la "S" se vuelve casi una rampa suave. A 9 horas le da apenas ≈ 0.69 (contra ≈ 0.98 de los otros): el modelo **no se atreve** a opinar (subajuste).
- `C=1` y `C=1000` → curvas casi idénticas (w ≈ 0.96 y ≈ 0.98). Con una sola característica y 200 datos, poco castigo ya basta y quitarlo del todo no cambia casi nada.

Con muchas características y pocos datos, un `C` grande sí produce coeficientes exagerados y sobreajuste. Por eso se elige con validación cruzada.
</details>

### 📖 `class_weight`: cuando la clase importante es rara
Si solo el 5 % de los casos son positivos (fraudes, enfermedades raras…), el modelo aprende que casi siempre es seguro decir "no". La pérdida está dominada por la clase mayoritaria.

`class_weight="balanced"` le dice: **cada error en la clase rara cuenta más**, en proporción inversa a su frecuencia. Es como mover el umbral, pero **durante el entrenamiento**.

### 🔮 Ejercicio 10 — Fraudes raros
Volvemos a las 2,000 transacciones de la tienda, con solo **≈4 % de fraudes**. Esta vez **no** movemos el umbral: usamos `predict` (umbral 0.5). Antes de ejecutar:
1. ¿Qué exactitud tendrá el modelo normal? ¿Y su *recall* de fraudes?
2. ¿Qué cambiará con `class_weight="balanced"`?

✍️ *Mis predicciones:*

In [ ]:
for peso in [None, "balanced"]:
    pred = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(class_weight=peso)), Xf, yf, cv=5)
    print(f"class_weight={str(peso):<9} → exactitud {np.mean(pred == yf):.1%} · "
          f"recall fraudes {recall_score(yf, pred):.0%} · precisión {precision_score(yf, pred, zero_division=0):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Exactitud **altísima** (≈96 %)… porque casi nunca dice "fraude". Su *recall* es bajísimo (≈6 %): no sirve para lo que lo queríamos. Un modelo que dijera siempre "no es fraude" tendría casi la misma exactitud (Lección 4.7).
2. Con `"balanced"` el *recall* salta a ≈57 %. A cambio, la exactitud baja (≈73 %) y la precisión se desploma (≈8 %): de cada 12 alertas, solo ~1 es fraude real. **Ya no hay un modelo "mejor": hay un compromiso que depende de los costos.**

> 💡 `class_weight` y mover el umbral son dos caminos hacia el mismo objetivo. El umbral tiene una ventaja: puedes ajustarlo **sin reentrenar**.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| `ConvergenceWarning` | El optimizador no llegó al fondo: datos sin escalar o `max_iter` bajo | Escalar con `StandardScaler` en un pipeline; subir `max_iter` |
| Exactitud alta pero *recall* casi 0 en la clase importante | Clases desbalanceadas + umbral 0.5 | `class_weight="balanced"`, bajar el umbral, medir con *recall*/precisión (Lección 4.2) |
| Exactitud ≈ 50 % en un problema que parece fácil | La frontera real no es una recta | Crear características o usar un modelo no lineal (5.3–5.5) |
| Coeficientes gigantes y probabilidades de exactamente 0 o 1 | Clases perfectamente separables o `C` muy grande | Bajar `C` (más regularización) |
| Elegiste el umbral mirando el conjunto de prueba | Fuga de información (Lección 4.6) | Elegirlo con `cross_val_predict` en el entrenamiento |
| Comparar coeficientes de variables con unidades distintas | Igual que en la Lección 5.1 | Estandarizar antes de comparar |

### 🔮 Ejercicio 11 — El aviso que todos ignoran
Entrenamos la logística con los datos de tumores **sin escalar** (hay columnas que van de 0 a 0.3 y otras de 0 a 4,000) y con el `max_iter` por defecto (100). ¿Qué crees que pasará?

✍️ *Mi predicción:*

In [ ]:
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    sin_escalar = LogisticRegression().fit(Xc_tr, yc_tr)
print("Avisos:", [a.category.__name__ for a in avisos])
print(f"Sin escalar → exactitud en entrenamiento: {sin_escalar.score(Xc_tr, yc_tr):.1%}")

con_escala = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xc_tr, yc_tr)
print(f"Con escala  → exactitud en entrenamiento: {con_escala.score(Xc_tr, yc_tr):.1%}  "
      f"(pasos usados: {con_escala[-1].n_iter_[0]} de 100)")

<details><summary><b>👀 Ver respuesta</b></summary>

Aparece un **`ConvergenceWarning`**: el optimizador agotó sus 100 pasos sin llegar al fondo de la montaña. El modelo funciona "más o menos", pero no es el que debería ser. Con los datos **escalados** converge en unos pocos pasos (menos de 20) y aprende mejor.

Regla práctica: **escala siempre** antes de una regresión logística (dentro de un pipeline, para no tener fuga de datos) y **no ignores** los avisos de convergencia.
</details>

---
## 8. 📋 Ficha del modelo

| | Regresión logística |
|---|---|
| **Tipo** | Supervisado: clasificación (binaria y multiclase) |
| **Intuición** | Una suma ponderada de las características, "aplastada" por una S para dar una probabilidad |
| **Cómo aprende** | Minimiza la entropía cruzada (castiga la seguridad equivocada) con un optimizador iterativo |
| **Hiperparámetros clave** | `C` (inverso de la regularización), `class_weight`, `max_iter`; y el **umbral** de decisión, que eliges después |
| **✅ Fortalezas** | Da probabilidades bien calibradas · Interpretable (signo y tamaño de coeficientes) · Rápida · Pocos datos bastan · Excelente línea base |
| **❌ Debilidades** | Frontera lineal (salvo que crees características) · Necesita escalar · Sensible a variables muy correlacionadas |
| **Úsalo cuando…** | Necesitas probabilidades para decidir con costos, quieres explicar el efecto de cada variable o buscas una línea base sólida |
| **Evítalo cuando…** | La frontera es muy no lineal y no sabes qué características crear (usa SVM con *kernel*, árboles o ensambles) |

## 9. 🧠 Resumen en 5 puntos
1. La regresión logística calcula un **puntaje** lineal y lo convierte en **probabilidad** con la sigmoide.
2. Aprende minimizando la **entropía cruzada**: equivocarse con seguridad cuesta muchísimo.
3. Su frontera de decisión es **lineal**; para fronteras curvas hay que crear características.
4. `predict` usa un umbral de 0.5, que solo es correcto si los dos errores cuestan igual: el umbral óptimo es $C_{FP}/(C_{FP}+C_{FN})$, y se comprueba con `cross_val_predict(..., method="predict_proba")`.
5. Sus perillas: `C` (al revés que `alpha`), `class_weight="balanced"` para clases raras y `max_iter`; y **siempre escala** los datos.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈35 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** Una regresión logística le da a un cliente un puntaje de **−3**. ¿Qué probabilidad aproximada de la clase 1 le asigna?
- a) −3 % · b) ≈ 5 % · c) 50 % · d) ≈ 95 %

**2.** Tu modelo tiene `ConvergenceWarning`. ¿Qué **no** ayuda?
- a) Escalar los datos · b) Subir `max_iter` · c) Bajar el umbral de decisión · d) Usar un pipeline con `StandardScaler`

**3.** Dejar pasar a un mal pagador cuesta 3 y rechazar a un buen pagador cuesta 1. ¿Umbral teórico para "rechazar"?
- a) 0.75 · b) 0.5 · c) 0.33 · d) 0.25

**4.** Subes `C` de 0.01 a 100. ¿Qué esperas?
- a) Más regularización y probabilidades más cercanas a 0.5
- b) Menos regularización, coeficientes más grandes, mayor riesgo de sobreajuste
- c) Que el umbral cambie a 0.01
- d) Nada: `C` solo afecta la velocidad

**5.** ¿Por qué se usa `cross_val_predict(..., method="predict_proba")` para elegir el umbral?
- a) Porque es más rápido que `predict`
- b) Porque da probabilidades de ejemplos que el modelo no vio, sin tocar el conjunto de prueba
- c) Porque calibra automáticamente las probabilidades
- d) Porque elige el umbral solo

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** sigmoide(−3) ≈ 0.047. Un puntaje negativo siempre da menos de 50 %; nunca una probabilidad negativa.
2. **c)** El umbral se aplica **después** de entrenar; no afecta al optimizador.
3. **d)** $1 / (1 + 3) = 0.25$.
4. **b)** `C` es el **inverso** de la regularización.
5. **b)** Probabilidades "honestas" de ejemplos no vistos, usando solo el entrenamiento. Así el conjunto de prueba queda intacto para el examen final.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Dos modelos tienen **la misma exactitud** (90 %) en un problema de detectar enfermos. El modelo A da probabilidades como 0.51, 0.49, 0.55… y el B da 0.02, 0.97, 0.99… ¿Cuál preferirías si después vas a **mover el umbral** según costos? ¿Qué información extra necesitarías?

**B. Caso de análisis 💼** Una aseguradora usa una regresión logística para decidir qué reclamos revisar por posible fraude. El gerente pide: *"Quiero que el modelo tenga la mayor exactitud posible"*. Sabes que solo 2 % de los reclamos son fraudulentos, que revisar uno cuesta $50 y que un fraude no detectado cuesta en promedio $2,000. ¿Qué le responderías? ¿Qué umbral propondrías como punto de partida?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** La exactitud con umbral 0.5 no dice nada de cómo se comportan las probabilidades. Para mover el umbral te interesa un modelo cuyas probabilidades **ordenen bien** a los casos (AUC, Lección 4.3) y estén **calibradas** ("70 %" significa 7 de cada 10). A tiene probabilidades tímidas, B muy seguras: ninguna de las dos es mala por sí misma. Necesitarías comprobar la calibración (¿de los casos con 0.9, cuántos son positivos?) y comparar costos con validación cruzada para varios umbrales.

**B.** Con 2 % de fraudes, el modelo "nunca revisar nada" tiene **98 % de exactitud** y no detecta ningún fraude: la exactitud es la métrica equivocada (Lección 4.8). Hay que minimizar el **costo**. Umbral teórico: $50 / (50 + 2000) ≈$ **0.024**: revisar todo reclamo con más de ~2.4 % de probabilidad de fraude. Luego comprobarlo con `cross_val_predict` y ver si el equipo de revisión da abasto con tantas revisiones.
</details>

### 10.3 🧩 Práctica integradora: "¿A quién le ofrezco el descuento?"

Una plataforma de streaming quiere evitar que sus suscriptores cancelen. Tiene una oferta de retención (un mes gratis):
- Darle la oferta a alguien que **no iba a cancelar** cuesta **1** (un mes regalado sin necesidad).
- **No** dársela a alguien que sí cancela cuesta **3** (se pierde el cliente).

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng_s = np.random.default_rng(21)
n_s = 3_000
subs = pd.DataFrame({
    "meses_suscrito":   rng_s.integers(1, 60, n_s),
    "horas_semana":     rng_s.gamma(2, 3, n_s).round(1),
    "perfiles":         rng_s.integers(1, 6, n_s),
    "quejas_ultimo_mes": rng_s.poisson(0.4, n_s),
    "precio":           rng_s.choice([99, 149, 199, 249], n_s),
})
z = (-0.6 - 0.04 * subs.meses_suscrito - 0.18 * subs.horas_semana - 0.25 * subs.perfiles
     + 0.9 * subs.quejas_ultimo_mes + 0.008 * subs.precio)
subs["cancela"] = (rng_s.random(n_s) < sigmoide(z)).astype(int)

X_s, y_s = subs.drop(columns="cancela"), subs.cancela
X_s_tr, X_s_te, y_s_tr, y_s_te = train_test_split(X_s, y_s, test_size=0.25, stratify=y_s, random_state=0)
print(f"Cancelan: {y_s.mean():.0%}  ·  entrenamiento: {len(X_s_tr)}  ·  prueba: {len(X_s_te)}")
subs.head()

**Paso 2 🔮 — Las estrategias "tontas".** Antes de cualquier modelo, calcula **en papel** el costo promedio por suscriptor de:
- **A)** no darle la oferta a nadie;
- **B)** dársela a todos.

Pista: usa el porcentaje de cancelaciones del Paso 1. Luego comprueba con la celda.

✍️ *Mis cálculos:*

In [ ]:
COSTO_FN, COSTO_FP = 3, 1     # perder un cliente / regalar un mes sin necesidad

print(f"A) Oferta a nadie : costo {costo_promedio(y_s_tr, np.zeros(len(y_s_tr), dtype=int)):.3f}")
print(f"B) Oferta a todos : costo {costo_promedio(y_s_tr, np.ones(len(y_s_tr), dtype=int)):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con ≈18 % de cancelaciones:
- A) Perdemos a todos los que cancelan: 3 × 0.18 ≈ **0.55**.
- B) Regalamos un mes a todos los que no iban a cancelar: 1 × 0.82 ≈ **0.82**.

Cualquier modelo que propongamos debe costar **menos de ≈0.55**; si no, sería mejor no hacer nada.
</details>

**Paso 3 ▶️ — El modelo con umbral 0.5.** Pipeline con escalado + logística, evaluado con validación cruzada (solo entrenamiento).

In [ ]:
pipe = make_pipeline(StandardScaler(), LogisticRegression())
prob_s = cross_val_predict(pipe, X_s_tr, y_s_tr, cv=5, method="predict_proba")[:, 1]

pred_05 = (prob_s >= 0.5).astype(int)
print(f"Umbral 0.5 → exactitud {np.mean(pred_05 == y_s_tr):.1%} · recall {recall_score(y_s_tr, pred_05):.0%} "
      f"· costo {costo_promedio(y_s_tr, pred_05):.3f}")

✍️ *¿Le gana a la mejor estrategia "tonta"? ¿Qué problema ves?*

<details><summary><b>👀 Ver respuesta</b></summary>

Exactitud ≈83 %, pero *recall* de apenas ≈20 %: con umbral 0.5 el modelo solo avisa cuando está muy seguro. Su costo (≈0.46) ya le gana a "nadie" (≈0.55), pero deja escapar a 4 de cada 5 clientes que se van. El umbral 0.5 supone que los dos errores cuestan igual, y aquí no es así.
</details>

**Paso 4 ✋ — Calcula el umbral teórico y compruébalo.** Escribe tu umbral en `UMBRAL` y ejecuta. La gráfica muestra el costo para todos los umbrales.

In [ ]:
UMBRAL = 0.25     # ✏️ escribe aquí el umbral que calculaste con la fórmula

umbrales = np.arange(0.05, 0.71, 0.01)
costos = [costo_promedio(y_s_tr, (prob_s >= u).astype(int)) for u in umbrales]
plt.plot(umbrales, costos); plt.axvline(UMBRAL, ls="--", c="g", label=f"Mi umbral = {UMBRAL}")
plt.xlabel("Umbral"); plt.ylabel("Costo promedio"); plt.legend(); plt.show()

pred_u = (prob_s >= UMBRAL).astype(int)
print(f"Umbral {UMBRAL} → exactitud {np.mean(pred_u == y_s_tr):.1%} · recall {recall_score(y_s_tr, pred_u):.0%} "
      f"· costo {costo_promedio(y_s_tr, pred_u):.3f}")
print(f"Mínimo de la curva: umbral {umbrales[np.argmin(costos)]:.2f} → costo {min(costos):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Umbral teórico: $1 / (1 + 3) =$ **0.25**. El costo baja de ≈0.46 a ≈0.38 y el *recall* casi se triplica (≈57 %). Fíjate: **la exactitud baja** (≈77 %) y aun así el negocio gana. El mínimo de la curva está muy cerca (umbral ≈0.28, costo ≈0.37): la teoría y los datos coinciden.
</details>

**Paso 5 ✏️ — ¿Y `class_weight="balanced"`?** Prueba el otro camino: pesos balanceados con el umbral por defecto (0.5).

In [ ]:
pipe_bal = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced"))
pred_bal = cross_val_predict(pipe_bal, X_s_tr, y_s_tr, cv=5)
print(f"Balanceado (umbral 0.5) → exactitud {np.mean(pred_bal == y_s_tr):.1%} · "
      f"recall {recall_score(y_s_tr, pred_bal):.0%} · costo {costo_promedio(y_s_tr, pred_bal):.3f}")

# ✋ ¿Qué variables empujan a cancelar? (coeficientes estandarizados)
coefs = pd.Series(pipe.fit(X_s_tr, y_s_tr)[-1].coef_[0], index=X_s.columns).sort_values()
coefs.round(2)

✍️ *¿Qué camino prefieres? ¿Qué variables aumentan el riesgo de cancelar?*

<details><summary><b>👀 Ver respuesta</b></summary>

- El modelo balanceado marca a muchos más clientes (*recall* ≈ 72 %) y su costo (≈0.385) es prácticamente igual al del umbral 0.25 (≈0.384). Aquí coinciden por casualidad: "balanced" compensa la **proporción** de clases (82/18), no **nuestra** regla de costos; con otra regla (p. ej. 10 a 1) ya no coincidirían. El umbral tiene la ventaja de que se ajusta sin reentrenar y se explica con la regla 3 a 1.
- Coeficientes positivos (aumentan el riesgo): **quejas en el último mes** (+0.59) y **precio** (+0.33). Negativos (protegen): meses suscrito (−0.70), horas de uso (−0.63) y número de perfiles (−0.39). Como están estandarizados, sí se pueden comparar entre sí. Accionable: atender rápido las quejas y cuidar a los clientes con planes caros que usan poco el servicio.
</details>

**Paso 6 ▶️ — Examen final, una sola vez.** Entrenamos con todo el entrenamiento y evaluamos en prueba con el umbral elegido.

In [ ]:
UMBRAL_FINAL = 0.25    # ✏️ el umbral que elegiste
final = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_s_tr, y_s_tr)
pred_te = (final.predict_proba(X_s_te)[:, 1] >= UMBRAL_FINAL).astype(int)

print(f"Modelo (umbral {UMBRAL_FINAL}) → costo {costo_promedio(y_s_te, pred_te):.3f} · recall {recall_score(y_s_te, pred_te):.0%}")
print(f"Oferta a nadie           → costo {costo_promedio(y_s_te, np.zeros(len(y_s_te), dtype=int)):.3f}")
print(f"Oferta a todos           → costo {costo_promedio(y_s_te, np.ones(len(y_s_te), dtype=int)):.3f}")
print(f"De cada 10 ofertas, {10 * precision_score(y_s_te, pred_te):.0f} van a clientes que sí iban a cancelar")

✋ Escribe un mensaje de **3 líneas** para la gerente de marketing, sin tecnicismos.

✍️ *Mi mensaje:*

<details><summary><b>👀 Ver un ejemplo</b></summary>

En prueba el modelo cuesta ≈0.36 por suscriptor, contra ≈0.55 de no hacer nada y ≈0.82 de dárselo a todos (≈ un tercio menos que no hacer nada), y detecta a ≈6 de cada 10 clientes que se iban a ir. La mitad de las ofertas llega a quien sí iba a cancelar. Un mensaje posible:

> *"Propongo ofrecer el mes gratis solo a los suscriptores con más de 25 % de riesgo de cancelar. Así llegamos a 6 de cada 10 clientes que se iban a ir, y cuesta un tercio menos que no hacer nada. Las quejas recientes, los planes caros y el poco uso son las principales señales de alerta."*
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "Logistic Regression" y "Logistic Regression Details" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario: *Logistic regression* y *Tuning the decision threshold for class prediction* (`TunedThresholdClassifierCV`, que automatiza la búsqueda del umbral).
- 📘 **Calibración:** scikit-learn, *Probability calibration* (`CalibratedClassifierCV`): cómo comprobar si "70 %" significa de verdad 70 %.

---
⬅️ **Anterior:** [5.1 Regresión lineal: la recta que mejor se ajusta](5.1-regresion-lineal.ipynb) · ➡️ **Siguiente:** [5.3 K vecinos más cercanos (KNN): "dime con quién andas"](5.3-k-vecinos-mas-cercanos.ipynb)